# 01. Image Classification - MobileNetV2

| 항목 | 내용 |
|---|---|
| 대상 모델 | `MobileNetV2.dxnn` |
| 카테고리 | **Image Classification** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 모델 출력이 확률이 아니라 점수라는 근거를 실행 결과에서 두 개 이상 제시할 수 있다.
2. 점수를 확률로 바꾸는 softmax 를 실제 숫자를 넣어 계산할 수 있다.
3. class index 와 이름표 목록이 어긋날 때 나타나는 증상을 설명할 수 있다.
4. 부류가 1,000개일 때 1위 확률이 10% 에 못 미치는 이유를 설명할 수 있다.

## 1. Image Classification (이미지 분류)

**사진 한 장 전체에 미리 정해진 이름표 하나를 붙이는 작업**

모델은 부류마다 점수를 하나씩 내고 그중 높은 쪽을 답으로 고른다. 대상이 사진 어디에 있는지는 계산하지
않으므로 사진 전체를 모델 입력 크기로 줄여 넣어도 답이 나온다.

**출력의 성격**

답이 이름표 하나이므로 좌표도 영역도 없다. 한 장에 대상이 여럿이어도 하나로 요약되고, 어느 픽셀이
무엇인지는 구분하지 못한다.

**활용 사례**

- 사진 자동 분류·앨범 정리
- 불량 유형 판정
- 검출로 잘라 낸 영역의 세부 종류 판정

**포함되지 않는 작업**

- 대상의 위치·개수 판정 (Object Detection)
- 픽셀 단위 영역 구분 (Semantic Segmentation)
- 같은 개체인지 대조 (Face Embedding · Person Re-ID)

## 2. 전체 파이프라인

```text
sample_dog.jpg  1200×900 BGR
  → 전처리         BGR→RGB · 224×224 직접 축소 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → Public Output  536  [1, 1000]  float32
  → 후처리         flatten → softmax → 내림차순 정렬 → 상위 top_k
  → Result         ClassificationResult 5개 (class_id · confidence)
  → 시각화         원본 위에 이름표와 확률을 글자로 얹음
```

![01. Image Classification 전체 파이프라인](resources/01_Image_Classification_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input.1` |
| Shape / Dtype | `[1, 224, 224, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**다른 분류 모델과의 차이점**

- **`uint8` 입력** : 0~1 로 나누는 정규화 없이 픽셀 값을 그대로 넣는다
- **비율 유지 없음** : 되돌릴 좌표가 없어 찌그러져도 답이 나온다

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `536` | `[1, 1000]` / `float32` | 1,000개 부류의 점수 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시로, Public 이 아닌 tensor 는
모델 내부 계산에만 쓰이고 밖으로 나오지 않는다. `536` 은 컴파일러가 붙인 식별자일 뿐이며 이름만으로
의미나 순서를 단정할 수 없다.

- shape 각 자리의 뜻 :

    | index | 값 | 뜻 |
    |---|---:|---|
    | 0 | 1 | 한 번에 처리하는 사진 장수 |
    | 1 | 1000 | 부류 개수. 각 자리가 부류 하나 |

    `float32` 하나가 4바이트이므로 1,000개는 4,000 바이트다.

- 출력은 확률이 아닌 점수 :

    - 음수가 섞여 있음 (1,000개 중 565개)
    - 다 더해도 1 이 아님 (합 : `0.0738`)
    - 크기 자체에는 뜻이 없고 서로 간의 대소만 뜻이 있음

- 속성 :

    모델이 반환하는 것은 번호뿐이다.
    
    273번이 `dingo` 라는 것은 모델 파일이 아니라 실행 코드가 들고 있는 ImageNet 이름표 배열이 정한다.

    전체 목록: [ImageNet class map](https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/)

    | index | 이름 |
    |---:|---|
    | 0 | tench |
    | 1 | goldfish |
    | 174 | Norwegian Elkhound |
    | 207 | Golden Retriever |
    | 248 | husky |
    | 250 | Siberian Husky |
    | 253 | Basenji |
    | 273 | dingo |
    | 998 | ear of corn |
    | 999 | toilet paper |


#### 출력을 사람이 읽는 값으로 바꾸기

점수 `z` 에 지수를 취해 전체 합으로 나누면 전부 양수가 되고 합이 1 이 된다. 지수 함수가 단조 증가라
순위는 바뀌지 않는다.

```text
m = max(z)
p[i] = exp(z[i] - m) / Σ exp(z[j] - m)
```

최댓값 `m` 을 먼저 빼는 것은 `exp` 가 넘치는 것을 막기 위한 장치이며 결과를 바꾸지 않는다. 분자와
분모에 같은 수를 곱한 것과 같기 때문이다.

이번 실행의 1위로 직접 계산하면 `z = 4.5981` 이고 `m` 도 같은 값이라 분자는 `exp(0) = 1` 이다. 분모가
`15.05` 이므로 `p = 1 / 15.05 = 0.0664` 즉 6.64% 다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 위 표에 적은
값이 실제 파일에 들어 있는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "MobileNetV2.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `top_k` | 5 | 5 | 확률 순으로 남길 상위 결과 개수 |

모델은 언제나 1,000개 점수를 다 내놓고 `top_k` 는 그중 몇 개를 남길지만 정한다.

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 ImageNet 이름표 1,000개를 읽는다. 이후 셀은 여기서
만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_dog.jpg"
CONFIG_PATH = DX_APP / "src" / "python_example" / "classification" / "mobilenetv2" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.utility import get_labels  # noqa: E402

IMAGENET_LABELS = get_labels("imagenet1000")

print("config", CONFIG)
print("labels", len(IMAGENET_LABELS), "/ index 273 =", IMAGENET_LABELS[273])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델의 후처리는
  되돌릴 좌표가 없어 받기만 하고 읽지 않는다
- `ClassificationResult` — 후처리가 만들어 시각화로 넘기는 결과 객체
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class ClassificationResult:
    """후처리가 만들어 시각화로 넘기는 결과. 원본 대응: common/base/i_processor.py"""

    class_id: int = 0
    class_name: str = ""
    confidence: float = 0.0


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. `parse_model` 출력과 같아야 한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

1200×900 BGR 원본을 RGB 224×224 `uint8` 로 바꾼다. 비율을 맞추는 여백을 넣지 않으므로 축마다 축소
비율이 달라지고, 그 비율을 Context 에 축별로 적어 둔다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 224×224, 비율 유지 없음
- dtype — `uint8` 유지 (`float32` 로 승격되지 않음)

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `mobilenetv2_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `mobilenetv2_preprocessor_process` — 이미지 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def mobilenetv2_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def mobilenetv2_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    # 비율을 맞추는 여백이 없으므로 축마다 축소 비율이 다르다. pad 는 0 으로 남는다.
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. `scale_x` 와 `scale_y` 가 서로 다르게 나오는 것이 비율을 유지하지
않았다는 증거다.

In [ ]:
preprocessor_state = mobilenetv2_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = mobilenetv2_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f} pad=({ctx.pad_x}, {ctx.pad_y})")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 음수 개수 — 하나라도 있으면 확률일 수 없다
- 전체 합 — 1 에서 멀면 확률 분포가 아니다
- 점수 그대로의 최대 자리 — softmax 를 거친 뒤의 1위와 같아야 한다

In [ ]:
logits = np.asarray(outputs[0]).flatten()
notebook_describe_array("logits", logits)
print(f"negative count = {int((logits < 0).sum())} / {logits.size}")
print(f"raw sum        = {logits.sum():.4f}")
print(f"argmax index   = {int(logits.argmax())} ({IMAGENET_LABELS[int(logits.argmax())]})")

### 4.4 후처리

Public Output 하나를 받아 확률 상위 `top_k` 개의 Result 를 만든다. 순서는 flatten → softmax →
내림차순 정렬 → 앞에서 `top_k` 개이며 이 순서를 바꾸지 않는다. 출력이 하나뿐이라 `outputs[0]` 을
고르는데, 근거는 순서가 아니라 이 모델의 Public Output 이 `536` 하나라는 사실이다.

`class_name` 은 후처리가 빈 문자열로 두고 시각화 단계에서 채운다. 이름표가 계산에는 쓰이지 않고
표시할 때만 필요하기 때문이다.

#### 후처리 함수 정의

- `mobilenetv2_postprocessor_create` — `top_k` 를 state 에 담음
- `mobilenetv2_softmax` — 점수를 확률로 변환
- `mobilenetv2_postprocessor_process` — 상위 `top_k` 개 Result 생성

In [ ]:
def mobilenetv2_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"top_k": config.get("top_k", 5)}


def mobilenetv2_softmax(x: np.ndarray) -> np.ndarray:
    """점수를 확률로 바꾼다. 원본 대응: ClassificationPostprocessor._softmax()"""
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)


def mobilenetv2_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.asarray(outputs[0])
    logits = output.flatten() if output.ndim > 1 else output
    probabilities = mobilenetv2_softmax(logits)

    top_indices = np.argsort(probabilities)[::-1][:state["top_k"]]
    return [
        ClassificationResult(class_id=int(idx), confidence=float(probabilities[idx]), class_name="")
        for idx in top_indices
    ]

#### 후처리 실행

확인할 것:

- 결과 개수 — `top_k` 와 같은지
- 확률 순서 — 내림차순인지
- 1위 index — 점수 그대로 찾은 최대 자리와 같은지

상위 5개 확률을 다 더해도 1 에 못 미치는 것이 정상이며, 나머지 995개 부류가 남은 확률을 나눠 갖는다.

In [ ]:
postprocessor_state = mobilenetv2_postprocessor_create(CONFIG)
results = mobilenetv2_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"result count = {len(results)} (top_k={postprocessor_state['top_k']})")
for rank, res in enumerate(results, start=1):
    print(f"  #{rank} class_id={res.class_id:4d} confidence={res.confidence:.6f} "
          f"class_name={res.class_name!r} label={IMAGENET_LABELS[res.class_id]}")
print(f"top-{len(results)} probability sum = {sum(r.confidence for r in results):.4f}")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 원본 위에 이름표와 확률을 얹는다. 후처리가 비워 둔 `class_name` 이 채워지는 자리가
여기이며, `class_id` 를 이름표 목록의 자리 번호로 쓴다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본
폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def mobilenetv2_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    if not results:
        return output

    height, width = output.shape[:2]
    overlay = output.copy()
    cv2.rectangle(overlay, (10, 10), (min(width - 10, 400), 30 + len(results) * 25), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, output, 0.5, 0, output)

    y_offset = 30
    for rank, res in enumerate(results):
        label = (IMAGENET_LABELS[res.class_id] if res.class_id < len(IMAGENET_LABELS)
                 else f"class_{res.class_id}")
        text = f"#{rank + 1}: {label} ({res.confidence * 100:.1f}%)"
        cv2.putText(output, text, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (255, 255, 255), 1, cv2.LINE_AA)
        y_offset += 25

    return output

#### 결과 시각화

최종 결과를 inline 으로 표시한다. 글자가 그려지는 대상은 224×224 로 줄인 입력이 아니라 1200×900
원본이다.

In [ ]:
output_image = mobilenetv2_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Top-5 classification", figsize=(11, 8))